In [1]:
import sqlite3
import pandas as pd
import numpy as np

conn = sqlite3.connect('espn_cfb_2026.db')

pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)

,name
0,Teams
1,Games
2,Box_Scores
3,Player_Box_Scores
4,Raw_Boxscores
5,Scoreboard_Fetches
6,Player_Boxscore_Fetches
7,Team_Defensive_Stats
8,Team_Defense_Opponent_Stats
9,Raw_Plays


In [2]:
load_games = """
SELECT game_id, week, home_team_id, away_team_id, home_score, away_score, neutral_site
FROM Games
WHERE status = 'Final'
"""

games = pd.read_sql_query(load_games, conn)

print(games.shape)
display(games.head())
display(games.dtypes)

(752, 7)


,game_id,week,home_team_id,away_team_id,home_score,away_score,neutral_site
0,401864494,1,30,23,42,26,0
1,401856766,1,2628,153,10,15,1
2,401858202,1,258,152,34,8,0
3,401864577,1,2449,55,33,7,0
4,401866408,1,2199,16,28,17,0


game_id           str
week            int64
home_team_id      str
away_team_id      str
home_score      int64
away_score      int64
neutral_site    int64
dtype: object

In [3]:
load_teams = """
SELECT team_id, short_name, conference
FROM Teams
"""

teams = pd.read_sql_query(load_teams, conn)

teams["conference"].value_counts(dropna=False)

conference
NaN                          102
Big Ten Conference            18
Atlantic Coast Conference     17
Big 12 Conference             16
Southeastern Conference       16
American Conference           14
Sun Belt Conference           14
Mid-American Conference       13
Conference USA                10
Mountain West Conference      10
Pac-12 Conference              8
FBS Independents               2
Name: count, dtype: int64

In [4]:
fcs_ids = set(teams[teams["conference"].isna()]["team_id"])

games["home_team_id"] = games["home_team_id"].mask(games["home_team_id"].isin(fcs_ids), "FCS")
games["away_team_id"] = games["away_team_id"].mask(games["away_team_id"].isin(fcs_ids), "FCS")

new_row = pd.DataFrame([{"team_id": "FCS", "short_name": "FCS", "conference": "FCS"}])
teams = teams if "FCS" in teams["team_id"].values else pd.concat([teams, new_row], ignore_index=True)

teams.shape

(241, 3)

In [5]:
home_fcs = games["home_team_id"].eq("FCS")
away_fcs = games["away_team_id"].eq("FCS")

print("Home FCS games:", home_fcs.sum())
print("Away FCS games:", away_fcs.sum())
print("Games vs. FCS (either side):", (home_fcs | away_fcs).sum())
print("FCS vs. FCS (both sides):", (home_fcs & away_fcs).sum())

Home FCS games: 0
Away FCS games: 123
Games vs. FCS (either side): 123
FCS vs. FCS (both sides): 0


In [6]:
home_view = games.rename(columns={
	"home_team_id": "team_id",
	"away_team_id": "opp_id",
	"home_score": "points_for",
	"away_score": "points_against",
})

away_view = games.rename(columns={
	"away_team_id": "team_id",
	"home_team_id": "opp_id",
	"away_score": "points_for",
	"home_score": "points_against",
})

home_view["location"] = np.where(home_view["neutral_site"] == 1, "neutral", "home")
away_view["location"] = np.where(away_view["neutral_site"] == 1, "neutral", "away")

team_games = pd.concat([home_view, away_view], ignore_index=True)
team_games = team_games.drop(columns=["neutral_site"])

team_games["margin"] = team_games["points_for"] - team_games["points_against"]
team_games["win"] = team_games["margin"].gt(0).astype(int)

In [7]:
display(len(team_games))
display(team_games["game_id"].value_counts().unique())
display(team_games["margin"].sum())
display((team_games["margin"] == 0).sum())
display(team_games["location"].value_counts())

1504

array([2])

np.int64(0)

np.int64(0)

location
home       742
away       742
neutral     20
Name: count, dtype: int64

In [8]:
records = team_games.groupby("team_id").agg(
	games=("game_id", "count"),
	wins=("win", "sum"),
	losses=("win", lambda x: (x == 0).sum()),
	win_pct=("win", "mean")
)

display(records.shape)
display(records["games"].sum())
display(records.loc["FCS"])

(139, 4)

np.int64(1504)

games      123.00000
wins         6.00000
losses     117.00000
win_pct      0.04878
Name: FCS, dtype: float64

In [9]:
team_games["opp_win_pct"] = team_games["opp_id"].map(records["win_pct"])

records["sos"] = team_games.groupby("team_id")["opp_win_pct"].mean()

display(team_games["opp_win_pct"].isna().sum())
display(records.sort_values("sos", ascending=False))

np.int64(0)

,games,wins,losses,win_pct,sos
team_id,,,,,
127,10,3,7,0.3,0.660000
2509,10,3,7,0.3,0.634878
251,10,8,2,0.8,0.630000
2483,10,7,3,0.7,0.624878
194,10,7,3,0.7,0.610000
...,...,...,...,...,...
2751,10,7,3,0.7,0.390333
2649,10,8,2,0.8,0.383060
113,10,5,5,0.5,0.377938


In [10]:
ranking = records.join(teams.set_index("team_id"))
ranking = ranking[ranking["conference"] != "FCS"]

ranking = ranking.sort_values(["win_pct", "sos"], ascending=[False, False])
ranking["rank"] = range(1, len(ranking) + 1)
display(ranking.head(25)[["rank", "short_name", "conference", "wins", "losses", "win_pct", "sos"]])

,rank,short_name,conference,wins,losses,win_pct,sos
team_id,,,,,,,
254,1,Utah,Big 12 Conference,10,0,1.000000,0.484878
26,2,UCLA,Big Ten Conference,10,0,1.000000,0.460000
2390,3,Miami,Atlantic Coast Conference,9,1,0.900000,0.574878
61,4,Georgia,Southeastern Conference,9,1,0.900000,0.504878
84,5,Indiana,Big Ten Conference,9,1,0.900000,0.494878
68,6,Boise St,Pac-12 Conference,9,1,0.900000,0.474878
87,7,Notre Dame,FBS Independents,9,1,0.900000,0.473333
2641,8,Texas Tech,Big 12 Conference,9,1,0.900000,0.464878
154,9,Wake Forest,Atlantic Coast Conference,9,1,0.900000,0.444878


In [11]:
team_idx = {tid: pos for pos, tid in enumerate(records.index)}
n = len(team_idx)
n

139

In [12]:
C = np.zeros((n, n))
b = np.ones(n)

for row in team_games.itertuples():
	i = team_idx[row.team_id]
	j = team_idx[row.opp_id]

	C[i, i] += 1
	C[i, j] -= 1

	if row.win == 1:
		b[i] += 0.5
	else:
		b[i] -= 0.5

C += 2 * np.eye(n)
display(np.allclose(C, C.T))
display(b.sum())
display(C[team_idx["FCS"], team_idx["FCS"]])

True

np.float64(139.0)

np.float64(125.0)

In [ ]:
r = np.linalg.solve(C, b)

display(r.mean())
display(r.min(), r.max())
records["colley"] = r

colley_rank = records.join(teams.set_index("team_id"))
colley_rank = colley_rank[colley_rank["conference"] != "FCS"]

colley_rank = colley_rank.sort_values("colley", ascending=False)
colley_rank["rank"] = range(1, len(colley_rank) + 1)

display(colley_rank.head(25)[["rank", "short_name", "conference", "wins", "losses", "sos", "colley"]])

np.float64(0.5)

np.float64(0.0395708854263823)

np.float64(0.9160864023960666)

,rank,short_name,conference,wins,losses,sos,colley
team_id,,,,,,,
254,1,Utah,Big 12 Conference,10,0,0.484878,0.916086
26,2,UCLA,Big Ten Conference,10,0,0.460000,0.910921
251,3,Texas,Southeastern Conference,8,2,0.630000,0.895170
61,4,Georgia,Southeastern Conference,9,1,0.504878,0.881503
2390,5,Miami,Atlantic Coast Conference,9,1,0.574878,0.874201
333,6,Alabama,Southeastern Conference,8,2,0.590000,0.846906
87,7,Notre Dame,FBS Independents,9,1,0.473333,0.844479
84,8,Indiana,Big Ten Conference,9,1,0.494878,0.826284
2641,9,Texas Tech,Big 12 Conference,9,1,0.464878,0.794817


np.int64(28)